# 2F-AGRO · Olho na Folha — Treino YOLOv8-cls (Colab GPU)

Notebook **self-contained** pra treinar o classificador de pragas com GPU grátis.

**Antes de rodar:** `Ambiente de execução → Alterar o tipo de hardware → GPU (T4)`.

Ao final, baixa o `2fagro-folha-cls-v1.pt` — é só colocar em `models/` no repo.

In [ ]:
# 1) Dependências
!pip -q install ultralytics
import torch; print('CUDA disponível:', torch.cuda.is_available())

In [ ]:
# 2) Baixa só as 6 classes do PlantVillage (sparse checkout, sem login)
%cd /content
!rm -rf pv
!git clone --filter=blob:none --no-checkout --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git pv
%cd pv
!git sparse-checkout init --cone
!git sparse-checkout set raw/color/Tomato___Bacterial_spot raw/color/Tomato___Late_blight raw/color/Squash___Powdery_mildew raw/color/Grape___Black_rot 'raw/color/Corn_(maize)___Common_rust_' raw/color/Tomato___healthy
!git checkout
!for d in raw/color/*/; do echo "$(ls \"$d\" | wc -l) $d"; done

In [ ]:
# 3) Prepara o dataset (renomeia classes + split 80/20)
import os, random, shutil
random.seed(42)
SRC = '/content/pv/raw/color'
OUT = '/content/data/folha-cls'
MAX_PER_CLASS, VAL_SPLIT = 800, 0.2
SOURCE_TO_KEY = {
    'Tomato___Bacterial_spot': 'mancha_bacteriana_tomate',
    'Tomato___Late_blight': 'requeima_tomate',
    'Squash___Powdery_mildew': 'oidio_mofo_branco',
    'Grape___Black_rot': 'podridao_negra_uva',
    'Corn_(maize)___Common_rust_': 'ferrugem_milho',
    'Tomato___healthy': 'saudavel',
}
EXTS = ('.jpg', '.jpeg', '.png')
if os.path.exists(OUT): shutil.rmtree(OUT)
for origem, chave in SOURCE_TO_KEY.items():
    pasta = os.path.join(SRC, origem)
    imgs = [os.path.join(pasta, f) for f in os.listdir(pasta) if f.lower().endswith(EXTS)]
    random.shuffle(imgs); imgs = imgs[:MAX_PER_CLASS]
    n_val = int(len(imgs) * VAL_SPLIT)
    for split, conj in (('train', imgs[n_val:]), ('val', imgs[:n_val])):
        dst = os.path.join(OUT, split, chave); os.makedirs(dst, exist_ok=True)
        for p in conj: shutil.copy2(p, dst)
    print(f'{chave:28s} train={len(imgs)-n_val:4d} val={n_val:4d}')
print('OK ->', OUT)

In [ ]:
# 4) Treino com GPU
from ultralytics import YOLO
model = YOLO('yolov8n-cls.pt')
model.train(data='/content/data/folha-cls', epochs=40, imgsz=224, batch=64, device=0, name='folha-cls', seed=42)
metrics = model.val()
print('top1:', metrics.top1, '| top5:', metrics.top5)

In [ ]:
# 5) Salva e baixa o modelo final
import glob, shutil
best = max(glob.glob('runs/**/weights/best.pt', recursive=True), key=os.path.getmtime)
shutil.copy2(best, '2fagro-folha-cls-v1.pt')
print('best:', best)
from google.colab import files
files.download('2fagro-folha-cls-v1.pt')